In [10]:
import numpy as np
import pandas as pd
from scipy.stats import weightedtau

In [11]:
def compute_weighted_tau(if_df, tc_df):
    """
    Compute weighted Kendall Tau between FOIF and TracIn scores.

    Each score list is normalised by its maximum absolute value.
    Larger-magnitude samples receive larger weights.
    """
    if_df = if_df[["Train_ID", "Score"]].copy()
    tc_df = tc_df[["Train_ID", "Score"]].copy()

    # Normalise each method independently
    if_max = if_df["Score"].abs().max()
    tc_max = tc_df["Score"].abs().max()

    if_df["Score"] = if_df["Score"] / if_max
    tc_df["Score"] = tc_df["Score"] / tc_max

    merged = pd.merge(
        if_df,
        tc_df,
        on="Train_ID",
        how="inner",
        suffixes=("_IF", "_TC")
    )

    # Weight samples according to their average absolute score magnitude
    def weight_function(rank_position):
        return (
            np.abs(merged["Score_IF"].iloc[rank_position])
            + np.abs(merged["Score_TC"].iloc[rank_position])
        ) / 2

    tau, _ = weightedtau(
        merged["Score_IF"],
        merged["Score_TC"],
        weigher=weight_function,
        rank=None
    )

    return tau

In [12]:
def compute_top_overlap(if_df, tc_df, top_fraction=0.10):
    """
    Compute the overlap between the top fraction of FOIF and TracIn
    ranked training samples.
    """
    if_ranked = (
        if_df[["Train_ID", "Score"]]
        .sort_values("Score", ascending=False)
        .reset_index(drop=True)
    )

    tc_ranked = (
        tc_df[["Train_ID", "Score"]]
        .sort_values("Score", ascending=False)
        .reset_index(drop=True)
    )

    # Use the number of common samples
    common_ids = set(if_ranked["Train_ID"]) & set(tc_ranked["Train_ID"])
    k = int(len(common_ids) * top_fraction)

    if k < 1:
        raise ValueError("Top fraction produces an empty top-K set.")

    top_if = set(if_ranked.head(k)["Train_ID"])
    top_tc = set(tc_ranked.head(k)["Train_ID"])

    overlap = len(top_if & top_tc) / k

    return overlap

In [13]:
def evaluate_pair(if_path, tc_path, top_fraction=0.10):
    if_df = pd.read_csv(if_path)
    tc_df = pd.read_csv(tc_path)

    required_cols = {"Train_ID", "Score"}

    if not required_cols.issubset(if_df.columns):
        raise ValueError(f"{if_path} does not contain {required_cols}")

    if not required_cols.issubset(tc_df.columns):
        raise ValueError(f"{tc_path} does not contain {required_cols}")

    tau = compute_weighted_tau(if_df, tc_df)

    overlap = compute_top_overlap(
        if_df,
        tc_df,
        top_fraction=top_fraction
    )

    return tau, overlap

In [14]:
experiments = [
    # Dataset size
    {
        "Data Property": "Dataset size",
        "Setting": "2K samples",
        "IF File": "IF_Train_Set_2k.csv",
        "TC File": "TC_Train_Set_2k.csv"
    },
    # {
    #     "Data Property": "Dataset size",
    #     "Setting": "4K samples",
    #     "IF File": "IF_Train_Set_4k.csv",
    #     "TC File": "TC_Train_Set_4k.csv"
    # },
    {
        "Data Property": "Dataset size",
        "Setting": "6K samples",
        "IF File": "IF_Train_Set_6k.csv",
        "TC File": "TC_Train_Set_6k.csv"
    },
    # {
    #     "Data Property": "Dataset size",
    #     "Setting": "8K samples",
    #     "IF File": "IF_Train_Set_8k.csv",
    #     "TC File": "TC_Train_Set_8k.csv"
    # },
    {
        "Data Property": "Dataset size",
        "Setting": "10K samples",
        "IF File": "IF_Train_Set_10k.csv",
        "TC File": "TC_Train_Set_10k.csv"
    },

    # Feature dimensionality
    {
        "Data Property": "Feature dimensionality",
        "Setting": "10 features",
        "IF File": "IF_Feature_Column_10_00.csv",
        "TC File": "TC_Feature_Column_10_00.csv"
    },
    {
        "Data Property": "Feature dimensionality",
        "Setting": "32 features",
        "IF File": "IF_Feature_Column_10_22.csv",
        "TC File": "TC_Feature_Column_10_22.csv"
    },
    {
        "Data Property": "Feature dimensionality",
        "Setting": "52 features",
        "IF File": "IF_Feature_Column_10_42.csv",
        "TC File": "TC_Feature_Column_10_42.csv"
    },

    # Class imbalance
    {
        "Data Property": "Class imbalance",
        "Setting": "5:5",
        "IF File": "IF_5_5_class.csv",
        "TC File": "TC_5_5_class.csv"
    },
    {
        "Data Property": "Class imbalance",
        "Setting": "3:7",
        "IF File": "IF_7_3_class.csv",
        "TC File": "TC_7_3_class.csv"
    },
    {
        "Data Property": "Class imbalance",
        "Setting": "1:9",
        "IF File": "IF_9_1_class.csv",
        "TC File": "TC_9_1_class.csv"
    },

    {
        "Data Property": "Sparsity",
        "Setting": "1:0.1",
        "IF File": "IF_sp1_de01_run1.csv",
        "TC File": "TC_sp1_de01_run1.csv"
    },
    # {
    #     "Data Property": "Sparsity",
    #     "Setting": "2:0.05",
    #     "IF File": "IF_sp2_de005_run1.csv",
    #     "TC File": "TC_sp2_de005_run1.csv"
    # },
    {
        "Data Property": "Sparsity",
        "Setting": "3:0.01",
        "IF File": "IF_sp3_de001_run1.csv",
        "TC File": "TC_sp3_de001_run1.csv"
    },
    # {
    #     "Data Property": "Sparsity",
    #     "Setting": "5:0.001",
    #     "IF File": "IF_sp5_de0001_cons.csv",
    #     "TC File": "TC_sp5_de0001_cons.csv"
    # },
    {
        "Data Property": "Sparsity",
        "Setting": "10:0.0001",
        "IF File": "IF_sp10_de00001_cons.csv",
        "TC File": "TC_sp10_de00001_cons.csv"
    },
    

]

In [15]:
density_labels = pd.read_csv(
    "Large_Small_Density_sep3_labelIDs_var42.csv"
)

print(density_labels.columns)
print(density_labels.head())

Index(['id', 'label', 'cluster_id'], dtype='object')
   id  label cluster_id
0   1      0          0
1   2      0          0
2   3      0          0
3   4      0          0
4   5      1    1_dense


In [16]:
def evaluate_density_group(
    if_path,
    tc_path,
    density_label_path,
    cluster_value,
    top_fraction=0.10
):
    if_df = pd.read_csv(if_path)
    tc_df = pd.read_csv(tc_path)
    labels = pd.read_csv(density_label_path)
    labels = labels.rename(columns={"id": "Train_ID"})

    selected_ids = labels.loc[
        labels["cluster_id"] == cluster_value,
        "Train_ID"
    ]

    if_group = if_df[
        if_df["Train_ID"].isin(selected_ids)
    ].copy()

    tc_group = tc_df[
        tc_df["Train_ID"].isin(selected_ids)
    ].copy()

    tau = compute_weighted_tau(if_group, tc_group)

    overlap = compute_top_overlap(
        if_group,
        tc_group,
        top_fraction=top_fraction
    )

    return tau, overlap

In [17]:
density_labels = density_labels.rename(
    columns={"id": "Train_ID"}
)

In [18]:
ordinary_experiments = experiments[:-2]

results = []

for experiment in ordinary_experiments:
    tau, overlap = evaluate_pair(
        if_path=experiment["IF File"],
        tc_path=experiment["TC File"],
        top_fraction=0.10
    )

    results.append({
        "Data Property": experiment["Data Property"],
        "Setting": experiment["Setting"],
        "Weighted Kendall Tau": tau,
        "Top-10% Overlap": overlap
    })

In [19]:
density_path = "Large_Small_Density_sep3_labelIDs_var42.csv"

for setting, cluster_value in [
    ("Dense region", "1_dense"),
    ("Sparse region", "1_sparse")
]:
    tau, overlap = evaluate_density_group(
        if_path="IF_LargeSmall_Density_sep3_var42.csv",
        tc_path="TC_LargeSmall_Density_sep3_var42.csv",
        density_label_path=density_path,
        cluster_value=cluster_value,
        top_fraction=0.10
    )

    results.append({
        "Data Property": "Neighbourhood density",
        "Setting": setting,
        "Weighted Kendall Tau": tau,
        "Top-10% Overlap": overlap
    })

In [20]:
consistency_table = pd.DataFrame(results)

consistency_table[
    "Weighted Kendall Tau"
] = consistency_table[
    "Weighted Kendall Tau"
].round(4)

consistency_table[
    "Top-10% Overlap"
] = consistency_table[
    "Top-10% Overlap"
].round(4)

print(consistency_table)

             Data Property        Setting  Weighted Kendall Tau  \
0             Dataset size     2K samples                0.1756   
1             Dataset size     6K samples                0.2781   
2             Dataset size    10K samples                0.2325   
3   Feature dimensionality    10 features                0.4149   
4   Feature dimensionality    32 features                0.2948   
5   Feature dimensionality    52 features                0.2856   
6          Class imbalance            5:5                0.3916   
7          Class imbalance            3:7                0.4431   
8          Class imbalance            1:9                0.6383   
9                 Sparsity          1:0.1                0.0403   
10   Neighbourhood density   Dense region                0.7107   
11   Neighbourhood density  Sparse region                0.1944   

    Top-10% Overlap  
0            0.1750  
1            0.2783  
2            0.2340  
3            0.6212  
4            0.511